# VoltVision Motor Portfolio - Simulation & Three-Regime Pricing

Runs the cohort simulation and prices it under all three regimes side-by-side (`tariff` | `glm` | `glm+telematics`), then exports the simulation data to `data/` for the separate **analysis.ipynb** to consume. All validation / EDA / EV / pricing-progression / consolidated-report work lives in **analysis.ipynb**, not here.

The baseline `cohort_results` is the **raw simulated book (premium-independent labels)** - it is NOT tariff-priced by default. The three priced regimes live in `BOOKS`.

In [1]:
import os, time, json
import numpy as np
import pandas as pd
from motor_lib import *
from motor_lib import _encode_features, _require_simulated, _pricing_features, _HAS_JOBLIB
os.makedirs('data', exist_ok=True)
os.makedirs('images', exist_ok=True)
os.makedirs('report/figures', exist_ok=True)


In [2]:
# Build the initial cohort (single generator call)
df = generate_dataset(COHORT_CONFIG, seed=COHORT_CONFIG['seed'])
print('initial book rows:', len(df))


initial book rows: 10000


In [3]:
df['CLAIM_LAMBDA'] = df.apply(compute_claim_lambda, axis=1)
print('Claim frequency model (Poisson GLM) applied')
print(f"Mean lambda: {df['CLAIM_LAMBDA'].mean():.4f}")


Claim frequency model (Poisson GLM) applied
Mean lambda: 0.1786


In [4]:
# ============================================================================
# BASELINE RUN - simulate once, price all three regimes side-by-side
# cohort_results stays the RAW simulated book (no premium applied); each regime
# in BOOKS is priced explicitly. Tariff premium is NOT the default reference and
# is NOT required for GLM/telem training (models fit on simulated CLAIM_COUNT).
# ============================================================================
cohort_results = simulate_cohort(df, n_years=5, new_entrants_per_year=None)
print('cohort_results = RAW simulated book (premium-independent labels only).')

# --- Calibrate expense_loading so GLM/telem land in the 65-75% LR band ---
def _base_lr(test_book, method, train_seed):
    b = price_book(test_book, method, COHORT_CONFIG, train_seed=train_seed)
    return b['CLAIM_AMOUNT'].sum() / b['FINAL_PREMIUM_SST'].sum()

_glm_base = _base_lr(cohort_results, 'glm', MODEL_TRAIN_SEED)
_el = min(max(_glm_base / 0.70, 0.85), 1.60)
COHORT_CONFIG['expense_loading'] = round(float(_el), 4)
print(f'CALIBRATION: GLM base LR (el=1.0) = {_glm_base*100:.2f}% -> '
      f'expense_loading={COHORT_CONFIG["expense_loading"]:.4f} (target ~70% GLM/telem LR)')

# --- Three regimes, side-by-side (GLM/telem trained on MODEL_TRAIN_SEED, tested on TEST_SEED) ---
# Price the three regimes in parallel (thread backend; shares RAM).
BOOKS = price_parallel(cohort_results, ('tariff', 'glm', 'telem'),
                       COHORT_CONFIG, train_seed=MODEL_TRAIN_SEED)
print('Three regimes priced (side-by-side):', list(BOOKS.keys()))


Year 2026: 10000 active policies, claims: 1728, freq: 15.7%, avg NCD priced: 24.79%, retention: 88.0%


Year 2027: 13797 active policies, claims: 2442, freq: 16.0%, avg NCD priced: 29.86%, retention: 88.2%


Year 2028: 17313 active policies, claims: 2986, freq: 15.5%, avg NCD priced: 32.16%, retention: 88.7%


Year 2029: 20664 active policies, claims: 3489, freq: 15.2%, avg NCD priced: 33.55%, retention: 89.2%


Year 2030: 23893 active policies, claims: 4103, freq: 15.6%, avg NCD priced: 34.79%, retention: 89.2%

Simulation complete. Total records: 85667
Year range: 2026 - 2030
Simulation wall time: 1.1s
cohort_results = RAW simulated book (premium-independent labels only).


CALIBRATION: GLM base LR (el=1.0) = 92.39% -> expense_loading=1.3198 (target ~70% GLM/telem LR)


Three regimes priced (side-by-side): ['tariff', 'glm', 'telem']


In [5]:
# ===================== LR DIAGNOSTIC / SEED ROBUSTNESS =====================
def _lr(b): return b['CLAIM_AMOUNT'].sum()/b['FINAL_PREMIUM_SST'].sum()*100

print('=== Seed robustness: GLM in-sample vs out-of-sample ===')
glm_in  = price_book(cohort_results, 'glm', COHORT_CONFIG)
glm_out = price_book(cohort_results, 'glm', COHORT_CONFIG, train_seed=MODEL_TRAIN_SEED)
print(f'  GLM in-sample     LR: {_lr(glm_in):.2f}%')
print(f'  GLM out-of-sample LR (train {MODEL_TRAIN_SEED}, test {TEST_SEED}): {_lr(glm_out):.2f}%')

print('\n=== LR by coverage (tariff vs telem) ===')
for m in ('tariff','telem'):
    bc = BOOKS[m].groupby('COVERAGE_TYPE').apply(
        lambda d: d['CLAIM_AMOUNT'].sum()/d['FINAL_PREMIUM_SST'].sum()*100, include_groups=False)
    print(f'  {m}: ' + ', '.join(f'{k} {v:.1f}%' for k,v in bc.items()))

print('\n=== Model calibration: predicted vs actual claim frequency ===')
pricer = train_pricing(cohort_results, 'glm', COHORT_CONFIG, train_seed=MODEL_TRAIN_SEED)
X = _encode_features(cohort_results, pricer['features'])
pred = pricer['model'].predict(X)
act = cohort_results['CLAIM_COUNT']
print(f'  predicted total freq: {pred.sum():.1f} | actual total: {act.sum():.1f} | ratio {pred.sum()/act.sum():.3f}')

print('\n=== NCD sensitivity (tariff LR with NCD removed) ===')
tar0 = BOOKS['tariff'].copy(); tar0['NCD_LEVEL'] = 0
tar0_prem = price_book(tar0, 'tariff', COHORT_CONFIG)['FINAL_PREMIUM_SST']
print(f'  tariff LR (NCD=0): {tar0["CLAIM_AMOUNT"].sum()/tar0_prem.sum()*100:.2f}%  vs with NCD: {_lr(BOOKS["tariff"]):.2f}%')

print('\n=== expense_loading sweep (telem, out-of-sample) ===')
_el0 = COHORT_CONFIG['expense_loading']
for el in (1.0, 1.15, 1.4):
    COHORT_CONFIG['expense_loading'] = el
    b = price_book(cohort_results, 'telem', COHORT_CONFIG, train_seed=MODEL_TRAIN_SEED)
    print(f'  expense_loading={el}: telem LR={_lr(b):.2f}%')
COHORT_CONFIG['expense_loading'] = _el0
print(f'\n[restored calibrated expense_loading = {_el0}]')

=== Seed robustness: GLM in-sample vs out-of-sample ===


  GLM in-sample     LR: 72.38%
  GLM out-of-sample LR (train 20260818, test 42): 70.00%

=== LR by coverage (tariff vs telem) ===
  tariff: Comprehensive 59.5%, TPFT 98.5%, TPO 698.2%
  telem: Comprehensive 69.2%, TPFT 67.5%, TPO 81.5%

=== Model calibration: predicted vs actual claim frequency ===
  predicted total freq: 14042.9 | actual total: 14748.0 | ratio 0.952

=== NCD sensitivity (tariff LR with NCD removed) ===
  tariff LR (NCD=0): 49.63%  vs with NCD: 74.95%

=== expense_loading sweep (telem, out-of-sample) ===


  expense_loading=1.0: telem LR=92.44%
  expense_loading=1.15: telem LR=80.38%
  expense_loading=1.4: telem LR=66.03%

[restored calibrated expense_loading = 1.3198]


In [6]:
# ============================================================================
# EXPORT SIMULATION DATA to data/ for analysis.ipynb (raw sim + 3 priced books
# + initial cohort + config). Parquet preferred; falls back to pickle.
# ============================================================================
os.makedirs('data', exist_ok=True)
def _save(df_obj, base):
    try:
        df_obj.to_parquet(base + '.parquet')
    except Exception:
        df_obj.to_pickle(base + '.pkl')
_save(cohort_results, 'data/cohort_sim')
for _m in BOOKS:
    _save(BOOKS[_m], f'data/book_{_m}')
_save(df, 'data/df_initial')
with open('data/config.json', 'w', encoding='utf-8') as _f:
    json.dump(COHORT_CONFIG, _f, indent=2, default=str)
print('Exports written to data/:')
for _f in sorted(os.listdir('data')):
    print(f'  {_f}')


Exports written to data/:
  book_glm.parquet
  book_tariff.parquet
  book_telem.parquet
  cohort_sim.parquet
  config.json
  df_initial.parquet
